# 00 · Environment check

Confirms that the GPU node is ready before any real work: Python packages, CUDA, XGBoost device, free disk,
and which raw data files are present. This is the cheapest unit in the pipeline and the first one to run after
`scripts/setup_env.sh` and `scripts/fetch_data.sh`.

**How this notebook runs.** Never by hand in an editor. `scripts/run_nb.sh 00_env_check dataset=... seed=...` executes it
with papermill on the GPU node, writes the executed copy to `runs/00_env_check/<dataset>/<graph_builder>/<seed>/executed.ipynb`,
and the last cell writes a `DONE` marker. If the process dies, the wrapper writes `FAILED` and the next `run_stage.sh`
picks the unit up again; checkpoints inside the run directory mean little work is repeated.

In [ ]:
# Parameters (papermill overrides these; keep names and types)
dataset = "all"
seed = 0
graph_builder = "none"
run_dir = ""       # empty -> runs/<stage>/<dataset>/<graph_builder>/<seed>
force = 0          # 1 -> redo work even if checkpoints exist
quick = 0          # 1 -> subsampled data, few epochs; smoke test only

In [ ]:
import json
import shutil
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import ckpt, config, paths, viz
from src.eval import results

STAGE = "00_env_check"
warnings.filterwarnings("ignore")
seed = int(seed)
force, quick = bool(int(force)), bool(int(quick))
run_dir = Path(run_dir) if run_dir else paths.run_dir(STAGE, dataset, graph_builder, seed)
run_dir.mkdir(parents=True, exist_ok=True)
t0 = time.time()
print(f"stage={STAGE} dataset={dataset} graph_builder={graph_builder} seed={seed} quick={quick} force={force}")
print("run_dir:", run_dir)
pd.Series(paths.describe())

## Packages and devices

In [ ]:
import importlib
import platform
import sys

from src import device

info = {"python": sys.version.split()[0], "platform": platform.platform(), **device.summary()}
for mod in ["numpy", "pandas", "scipy", "sklearn", "xgboost", "torch", "torch_geometric", "pyg_lib",
            "papermill", "pyarrow", "matplotlib", "shap", "kaggle"]:
    try:
        info[f"v_{mod}"] = importlib.import_module(mod).__version__
    except Exception as exc:
        info[f"v_{mod}"] = f"MISSING ({type(exc).__name__})"
pd.Series(info)

## GPU smoke test

A 2048×2048 matmul on the GPU, and a five-round XGBoost fit on the device XGBoost will use.

In [ ]:
gpu_ok = False
if device.cuda_available():
    import torch

    a = torch.randn(2048, 2048, device="cuda")
    t = time.time()
    (a @ a).sum().item()
    torch.cuda.synchronize()
    info["gpu_matmul_ms"] = round((time.time() - t) * 1000, 2)
    info["gpu_mem_total_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    gpu_ok = True
    print("GPU ok:", info["gpu"], info["gpu_mem_total_gb"], "GB")
else:
    print("No GPU visible. GNN stages will run on CPU and be slow; XGBoost falls back to CPU.")

import xgboost as xgb

rng = np.random.default_rng(0)
bst = xgb.train({"device": device.xgb_device(), "tree_method": "hist", "objective": "binary:logistic"},
                xgb.DMatrix(rng.random((2000, 8)), label=rng.integers(0, 2, 2000)), num_boost_round=5)
info["xgb_device_used"] = device.xgb_device()
print("xgboost trained on", info["xgb_device_used"])

## Disk

Raw data is ~7 GB, run outputs up to ~10 GB. Anything under 20 GB free on `DATA_ROOT` or `RUNS_ROOT` deserves attention.

In [ ]:
rows = []
for name, p in [("DATA_ROOT", paths.DATA_ROOT), ("RUNS_ROOT", paths.RUNS_ROOT), ("REPO_ROOT", paths.REPO_ROOT)]:
    p.mkdir(parents=True, exist_ok=True)
    u = shutil.disk_usage(p)
    rows.append({"path": name, "location": str(p), "free_gb": round(u.free / 1e9, 1), "total_gb": round(u.total / 1e9, 1),
                 "used_pct": round(100 * u.used / u.total, 1)})
disk = pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(6, 2.6))
ax.barh(disk["path"], disk["total_gb"], color="lightgrey", label="total")
ax.barh(disk["path"], disk["free_gb"], color="C2", label="free")
ax.set(xlabel="GB", title="Disk")
ax.legend()
viz.savefig(fig, run_dir, "disk")
plt.show()
disk

## Raw data present?

Missing files: run `scripts/fetch_data.sh <dataset>`. DGraph-Fin must be downloaded by hand (licence click-through).

In [ ]:
from src.data import registry

rows = []
for ds in config.DATASETS:
    cfg = config.load_config(ds)
    present = registry.get(ds).raw_present(cfg)
    for f, ok in present.items():
        rows.append({"dataset": ds, "file": f, "present": ok})
    if not any(present.values()):
        print(f"[{ds}] missing ->", registry.get(ds).instructions())
raw = pd.DataFrame(rows)
raw

## Unit plan

Everything `run_all.sh` will execute, from `configs/stages.yaml`.

In [ ]:
status = results.unit_status()
status.groupby("stage")["status"].value_counts().unstack(fill_value=0)

In [ ]:
summary = {
    "gpu_ok": gpu_ok,
    "raw_present": raw.groupby("dataset")["present"].any().to_dict(),
    "free_gb": disk.set_index("path")["free_gb"].to_dict(),
    "elapsed_s": round(time.time() - t0, 1),
    **{k: v for k, v in info.items() if isinstance(v, (str, int, float, bool)) or v is None},
}
results.save_metrics(run_dir, {"models": {}, "env": summary})
ckpt.mark_done(run_dir, summary)
summary